# Neural architecture ablations: Siamese, bilinear, cross-attention, LoRA

**Status:** model and training scaffolds for review; no full training run.

**When to run:** after strict splits are frozen and classical baselines are
saved. Use a fresh GPU runtime for each encoder family.

**Inputs:** pooled caches for Siamese/bilinear models; residue-level padded
embeddings plus masks for cross-attention; raw sequences for LoRA.

**Outputs:** unique checkpoints and prediction tables only after the
training section is enabled.

**Usage:** begin with the shape tests. Then connect exactly one data adapter,
one model configuration, and one split protocol. Do not compare models that
use different splits, preprocessing, or model-selection budgets. Full
1,024-by-1,024 residue cross-attention is quadratic in memory; begin with
reviewed 256-residue windows or a chunked/sparse attention implementation.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and deterministic controls

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
from dataclasses import asdict, dataclass
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 44
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_MODEL = False
WRITE_OUTPUTS = False
RUN_TAG = "neural_ablation_v1"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)
ENCODER_TO_RUN = "ProtT5"  # One frozen cache per run; same SNOOPPI_SPLIT_DIR as 00/01/03.
MODEL_TO_RUN = "siamese"  # "siamese" or "bilinear"; residue/LoRA paths need separate adapters.
QUICK_MODE = True
SEEDS = [44] if QUICK_MODE else [41, 42, 43, 44, 45]
MAX_TRAIN_ROWS = 5_000 if QUICK_MODE else None


## 2. Shared symmetric pooled-embedding models

In [ ]:
class SymmetricPairFeatures(nn.Module):
    """Partner-swap invariant feature map used by pooled models."""

    def forward(self, embedding_a, embedding_b):
        return torch.cat([
            embedding_a + embedding_b,
            torch.abs(embedding_a - embedding_b),
            embedding_a * embedding_b,
        ], dim=-1)


class SiameseMLP(nn.Module):
    """Shared protein tower followed by the standard symmetric interaction head."""

    def __init__(self, input_dim, tower_dim=256, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.tower = nn.Sequential(
            nn.Linear(input_dim, tower_dim),
            nn.LayerNorm(tower_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.combine = SymmetricPairFeatures()
        self.head = nn.Sequential(
            nn.Linear(3 * tower_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, embedding_a, embedding_b):
        projected_a = self.tower(embedding_a)
        projected_b = self.tower(embedding_b)
        return self.head(self.combine(projected_a, projected_b)).squeeze(-1)


class LowRankBilinear(nn.Module):
    """Low-rank multiplicative interactions avoid a full D-by-D bilinear matrix."""

    def __init__(self, input_dim, rank=128, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.left = nn.Linear(input_dim, rank, bias=False)
        self.right = nn.Linear(input_dim, rank, bias=False)
        self.head = nn.Sequential(
            nn.LayerNorm(rank),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(rank, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, embedding_a, embedding_b):
        # Averaging both directions preserves partner-swap invariance even though
        # the two projections are learned separately.
        interaction_ab = self.left(embedding_a) * self.right(embedding_b)
        interaction_ba = self.left(embedding_b) * self.right(embedding_a)
        return self.head((interaction_ab + interaction_ba) / 2).squeeze(-1)

## 3. Residue-level bidirectional cross-attention

In [ ]:
class MaskedAttentionPool(nn.Module):
    def __init__(self, dimension):
        super().__init__()
        self.query = nn.Parameter(torch.zeros(dimension))
        nn.init.normal_(self.query, std=dimension ** -0.5)

    def forward(self, residues, valid_mask):
        if valid_mask.dtype != torch.bool or valid_mask.shape != residues.shape[:2] or not valid_mask.any(dim=1).all():
            raise ValueError("Residue masks must be boolean, match residue shapes, and retain at least one residue per protein")
        logits = torch.einsum("bld,d->bl", residues, self.query)
        logits = logits.masked_fill(~valid_mask, -torch.inf)
        weights = torch.softmax(logits, dim=-1)
        return torch.einsum("bl,bld->bd", weights, residues), weights


class ResidueCrossAttention(nn.Module):
    """Cross-attend A→B and B→A, then pool both directions symmetrically."""

    def __init__(self, input_dim, model_dim=256, heads=8, dropout=0.1):
        super().__init__()
        if model_dim % heads:
            raise ValueError("model_dim must be divisible by heads")
        self.project = nn.Linear(input_dim, model_dim)
        self.cross_attention = nn.MultiheadAttention(
            model_dim, heads, dropout=dropout, batch_first=True
        )
        self.pool = MaskedAttentionPool(model_dim)
        self.combine = SymmetricPairFeatures()
        self.head = nn.Sequential(
            nn.Linear(3 * model_dim, model_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(model_dim, 1),
        )

    def attend(self, query, key_value, query_mask, key_mask):
        attended, attention = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=~key_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        # Residual update retains the query protein's own residue representation.
        pooled, pool_weights = self.pool(query + attended, query_mask)
        return pooled, attention, pool_weights

    def forward(self, residues_a, mask_a, residues_b, mask_b, return_attention=False):
        for residues, mask in ((residues_a, mask_a), (residues_b, mask_b)):
            if mask.dtype != torch.bool or mask.shape != residues.shape[:2] or not mask.any(dim=1).all():
                raise ValueError("Invalid/all-padding residue mask")
            if not torch.isfinite(residues).all():
                raise ValueError("Residue embeddings must be finite, including padding")
        projected_a = self.project(residues_a)
        projected_b = self.project(residues_b)
        pooled_ab, attention_ab, pool_a = self.attend(projected_a, projected_b, mask_a, mask_b)
        pooled_ba, attention_ba, pool_b = self.attend(projected_b, projected_a, mask_b, mask_a)
        logits = self.head(self.combine(pooled_ab, pooled_ba)).squeeze(-1)
        if return_attention:
            return logits, {
                "A_to_B": attention_ab,
                "B_to_A": attention_ba,
                "pool_A": pool_a,
                "pool_B": pool_b,
            }
        return logits

## 4. Shape and symmetry tests

In [ ]:
batch_size, embedding_dim = 3, 128
embedding_a = torch.randn(batch_size, embedding_dim)
embedding_b = torch.randn(batch_size, embedding_dim)

for model in [
    SiameseMLP(embedding_dim),
    LowRankBilinear(embedding_dim),
]:
    model.eval()
    score_ab = model(embedding_a, embedding_b)
    score_ba = model(embedding_b, embedding_a)
    assert score_ab.shape == (batch_size,)
    assert torch.allclose(score_ab, score_ba, atol=1e-6), type(model).__name__
    print("PASS symmetry:", type(model).__name__)

residue_model = ResidueCrossAttention(input_dim=embedding_dim, model_dim=64, heads=4).eval()
residues_a = torch.randn(batch_size, 11, embedding_dim)
residues_b = torch.randn(batch_size, 7, embedding_dim)
mask_a = torch.ones(batch_size, 11, dtype=torch.bool)
mask_b = torch.ones(batch_size, 7, dtype=torch.bool)
mask_a[0, 8:] = False
mask_b[1, 5:] = False
with torch.no_grad():
    residue_score_ab = residue_model(residues_a, mask_a, residues_b, mask_b)
    residue_score_ba = residue_model(residues_b, mask_b, residues_a, mask_a)
assert torch.allclose(residue_score_ab, residue_score_ba, atol=1e-6)
print("PASS symmetry: ResidueCrossAttention")

## 5. Shared training loop

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    max_epochs: int = 50
    patience: int = 10
    gradient_clip_norm: float = 1.0


def run_epoch(model, loader, criterion, optimizer=None, gradient_clip_norm=1.0):
    training = optimizer is not None
    model.train(training)
    losses, labels_all, probabilities_all = [], [], []
    for batch in loader:
        # Data adapters should return positional model inputs followed by labels.
        *inputs, labels = batch
        inputs = [value.to(DEVICE, non_blocking=True) for value in inputs]
        labels = labels.to(DEVICE, non_blocking=True).float()
        with torch.set_grad_enabled(training):
            logits = model(*inputs)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
                optimizer.step()
        losses.append(loss.detach().cpu().item() * len(labels))
        labels_all.append(labels.detach().cpu().numpy())
        probabilities_all.append(torch.sigmoid(logits).detach().cpu().numpy())
    if not labels_all:
        raise ValueError("Cannot evaluate an empty loader")
    return {
        "loss": sum(losses) / sum(len(x) for x in labels_all),
        "labels": np.concatenate(labels_all),
        "probabilities": np.concatenate(probabilities_all),
    }


def fit_with_early_stopping(model, train_loader, validation_loader, config):
    from snooppi_utils import macro_average_precision

    if config.max_epochs < 1 or config.patience < 1 or config.gradient_clip_norm <= 0:
        raise ValueError("Training config epochs, patience and gradient clip must be positive")
    model = model.to(DEVICE)
    labels = train_loader.dataset.labels
    if set(np.unique(labels)) != {0, 1}:
        raise ValueError("Training requires both binary classes")
    positive_weight = (len(labels) - labels.sum()) / max(1, labels.sum())
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(positive_weight, device=DEVICE)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    best_state, best_score, patience_used, history = None, -np.inf, 0, []
    for epoch in range(1, config.max_epochs + 1):
        train_result = run_epoch(model, train_loader, criterion, optimizer, config.gradient_clip_norm)
        validation_result = run_epoch(model, validation_loader, criterion)
        validation_score = macro_average_precision(
            validation_result["labels"], validation_result["probabilities"]
        )
        history.append({
            "epoch": epoch,
            "train_loss": train_result["loss"],
            "validation_loss": validation_result["loss"],
            "validation_macro_ap": validation_score,
        })
        if validation_score > best_score:
            best_score = validation_score
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            patience_used = 0
        else:
            patience_used += 1
        if patience_used >= config.patience:
            break
    if best_state is None:
        raise ValueError("No finite validation macro-AP checkpoint was selected")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

### 5a. Optional pooled-cache training and common prediction export

Only this block runs training when `TRAIN_MODEL=True`. It loads the same reviewed
split set, selects checkpoints/thresholds on validation only, and exports the
same prediction identities as 01/03. Cross-attention and LoRA remain separate
scaffolds because pooled caches contain no residue information and cannot be
used to fine-tune encoders. Use a unique `RUN_TAG` for each encoder/model.


In [ ]:
from snooppi_utils import load_embedding_cache, select_threshold, summarize_predictions
from sklearn.model_selection import train_test_split

class PooledPairDataset(Dataset):
    def __init__(self, frame, cache):
        self.frame = frame.reset_index(drop=True)
        self.cache = cache
        self.labels = frame[LABEL].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return (torch.from_numpy(self.cache[row[SEQUENCE_A]]),
                torch.from_numpy(self.cache[row[SEQUENCE_B]]),
                torch.tensor(self.labels[index]))

if TRAIN_MODEL:
    splits = load_splits(PATHS)
    assert_model_ready(splits)
    manifest = split_manifest(PATHS)
    test_identity = prediction_identity(splits["test"], manifest)
    train_frame = splits["train"]
    if MAX_TRAIN_ROWS is not None and len(train_frame) > MAX_TRAIN_ROWS:
        train_frame, _ = train_test_split(train_frame, train_size=MAX_TRAIN_ROWS,
                                        stratify=train_frame[LABEL], random_state=44)
    selected_cache_path = cache_paths(PATHS)[ENCODER_TO_RUN]
    cache, cache_metadata = load_embedding_cache(selected_cache_path, ENCODER_TO_RUN)
    coverage = validate_cache_coverage(cache, splits)
    cache_records = {ENCODER_TO_RUN: {"path": str(selected_cache_path), "sha256": file_sha256(selected_cache_path),
                                    "metadata": cache_metadata, "coverage": coverage}}
    model_factories = {"siamese": SiameseMLP, "bilinear": LowRankBilinear}
    if MODEL_TO_RUN not in model_factories:
        raise ValueError("Pooled training supports siamese/bilinear only")
    config = TrainingConfig(max_epochs=3, patience=2) if QUICK_MODE else TrainingConfig()
    output_files = {"metrics": PATHS["tables"] / f"{RUN_TAG}_metrics.csv",
                    "predictions": PATHS["tables"] / f"{RUN_TAG}_test_predictions.csv",
                    "manifest": PATHS["tables"] / f"{RUN_TAG}_manifest.json"}
    checkpoint_paths = {seed: PATHS["checkpoints"] / f"{RUN_TAG}_seed{seed}.pt" for seed in SEEDS}
    if WRITE_OUTPUTS:
        from snooppi_utils import check_output_paths
        check_output_paths(list(output_files.values()) + list(checkpoint_paths.values()))
    metric_rows, prediction_tables = [], []
    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        generator = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(PooledPairDataset(train_frame, cache), batch_size=1024,
                                  shuffle=True, generator=generator)
        validation_loader = DataLoader(PooledPairDataset(splits["validation"], cache), batch_size=1024)
        test_loader = DataLoader(PooledPairDataset(splits["test"], cache), batch_size=1024)
        model = model_factories[MODEL_TO_RUN](next(iter(cache.values())).shape[0]).to(DEVICE)
        model, history = fit_with_early_stopping(model, train_loader, validation_loader, config)
        criterion = nn.BCEWithLogitsLoss()
        validation_result = run_epoch(model, validation_loader, criterion)
        threshold = select_threshold(validation_result["labels"], validation_result["probabilities"])
        test_result = run_epoch(model, test_loader, criterion)
        selected_epoch = int(history.loc[history.validation_macro_ap.idxmax(), "epoch"])
        metrics = summarize_predictions(test_result["labels"], test_result["probabilities"], threshold)
        metrics.update(encoder=ENCODER_TO_RUN, model=MODEL_TO_RUN, seed=seed, quick_mode=QUICK_MODE,
                       split_sha256=manifest["sha256"]["test"], selected_epoch=selected_epoch,
                       trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad))
        metric_rows.append(metrics)
        prediction_tables.append(test_identity.assign(encoder=ENCODER_TO_RUN, model=MODEL_TO_RUN, seed=seed,
                                  probability=test_result["probabilities"], threshold=threshold, quick_mode=QUICK_MODE))
        if WRITE_OUTPUTS:
            prepare_outputs([checkpoint_paths[seed]])
            torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        "model": MODEL_TO_RUN, "encoder": ENCODER_TO_RUN, "seed": seed,
                        "selected_epoch": selected_epoch, "threshold": threshold, "split_manifest": manifest,
                        "config": asdict(config), "history": history.to_dict("records")}, checkpoint_paths[seed])
    metrics_table = pd.DataFrame(metric_rows)
    predictions_table = pd.concat(prediction_tables, ignore_index=True)
    print(metrics_table.round(4).to_string(index=False))
    if WRITE_OUTPUTS:
        prepare_outputs(output_files.values())
        metrics_table.to_csv(output_files["metrics"], index=False)
        predictions_table.to_csv(output_files["predictions"], index=False)
        save_json(make_run_manifest(PATHS, splits, {"run_tag": RUN_TAG, "encoder": ENCODER_TO_RUN,
                  "model": MODEL_TO_RUN, "seeds": SEEDS, "quick_mode": QUICK_MODE,
                  "training_rows": len(train_frame), "training_config": asdict(config)}, cache_records), output_files["manifest"])
else:
    print("Pooled training disabled. Shape tests do not load data or caches.")


## 6. ProtT5 LoRA adapter draft

Run this cell only in a fresh GPU runtime after installing compatible
`transformers`, `peft`, and `sentencepiece` versions. Inspect the printed
trainable modules before training; target-module names can change across
library versions. Start with the strict split and small batches with
gradient accumulation.

In [ ]:
ENABLE_LORA_SETUP = False

if ENABLE_LORA_SETUP:
    from transformers import T5EncoderModel, T5Tokenizer
    from peft import LoraConfig, TaskType, get_peft_model

    model_id = "Rostlab/prot_t5_xl_half_uniref50-enc"
    tokenizer = T5Tokenizer.from_pretrained(model_id, do_lower_case=False)
    encoder = T5EncoderModel.from_pretrained(model_id, torch_dtype=torch.float16)
    lora_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q", "v"],
        bias="none",
    )
    encoder = get_peft_model(encoder, lora_config)
    encoder.print_trainable_parameters()
    target_names = [name for name, _ in encoder.named_modules() if name.endswith((".q", ".v"))]
    if not target_names:
        raise RuntimeError("No LoRA target modules matched; inspect encoder.named_modules()")
    print("Matched LoRA targets (first 20):", target_names[:20])
else:
    print("LoRA setup disabled. Review versions, GPU memory, and split protocol first.")

## 7. Required experiment matrix

Hold encoder, split, selection metric, seed list, and search budget fixed.

1. Existing symmetric MLP on pooled embeddings.
2. Siamese tower with matched parameter budget.
3. Low-rank bilinear model at several preregistered ranks.
4. Residue cross-attention with the same downstream hidden width.
5. Frozen ProtT5 versus LoRA ProtT5, recording trainable parameters, memory,
   wall time, and exact adapter configuration.

Save validation-selected checkpoints and row-level test predictions so
notebook 04 can compute paired uncertainty and calibration.